In [ ]:
# Install dependencies and connect Google Drive
# Install the latest Hugging Face libraries
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Mount Drive to save files securely
drive.mount('/content/drive')

# Create the project directory in Drive if it does not exist
PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PASTA_PROJETO, exist_ok=True)
print(f"Diretório de trabalho pronto em: {PASTA_PROJETO}")

In [ ]:
# Importe as bibliotecas necessárias
from google.colab import userdata
import os
from huggingface_hub import login # Importa a função login do Hugging Face Hub

# Carregue o token do Hugging Face dos segredos do Colab
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        # Tenta fazer o login com o token do Hugging Face
        login(token=hf_token, add_to_git_credential=False) # 'add_to_git_credential=False' para não solicitar credenciais git
        print("Token do Hugging Face carregado e configurado com sucesso via huggingface_hub.login().")
    else:
        print("AVISO: O segredo 'HF_TOKEN' foi encontrado, mas está vazio ou é None. Por favor, verifique o valor nos segredos do Colab.")
        # Se o token estiver vazio/None, ainda define a variável de ambiente como string vazia para evitar erros posteriores
        os.environ['HF_TOKEN'] = ''
except userdata.SecretNotFoundError:
    print("ATENÇÃO: O segredo 'HF_TOKEN' não foi encontrado. Por favor, adicione seu token do Hugging Face aos segredos do Colab.")
    os.environ['HF_TOKEN'] = '' # Garante que a variável de ambiente seja definida, mesmo que vazia
except Exception as e:
    print(f"Ocorreu um erro ao carregar ou configurar o token do Hugging Face: {e}")
    os.environ['HF_TOKEN'] = '' # Garante que a variável de ambiente seja definida, mesmo que vazia

In [ ]:
from datasets import load_dataset, concatenate_datasets
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2TokenizerFast
import os
import shutil

PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
pasta_tok = os.path.join(PASTA_PROJETO, "tokenizador")

# Verifica se o tokenizador já existe e o carrega, caso contrário, treina um novo.
if os.path.exists(pasta_tok) and os.path.isdir(pasta_tok) and \
   os.path.exists(os.path.join(pasta_tok, "vocab.json")) and \
   os.path.exists(os.path.join(pasta_tok, "merges.txt")):
    print(f"Tokenizador encontrado em: {pasta_tok}. Carregando tokenizador existente...")
    tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, local_files_only=True)
    print("Tokenizador carregado com sucesso!")
else:
    print(f"Tokenizador não encontrado ou incompleto em {pasta_tok}. Treinando novo tokenizador...")

    # 1. Carrega frações exatas direto para o cache local do Colab (SEM streaming=True)
    # Dividido proporcionalmente para somar 50.000 artigos no total (50% EN, 25% PT, 25% ES)
    print("Baixando fatias da Wikipédia para a memória local (Isso leva cerca de 1-2 minutos)...")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:25000]")
    wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train[:12500]")
    wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train[:12500]")

    # Junta os datasets baixados localmente e embaralha na memória RAM
    print("Misturando e preparando os dados...")
    dataset_misto = concatenate_datasets([wiki_en, wiki_pt, wiki_es])
    dataset_misto = dataset_misto.shuffle(seed=42)

    # Gerador usado para alimentar o treinador do tokenizador direto da memória RAM
    def extrair_texto():
        for item in dataset_misto:
            yield item["text"]

    print("Treinando o tokenizador... Agora deve levar de 2 a 3 minutos.")
    tokenizer_raw = ByteLevelBPETokenizer()
    tokenizer_raw.train_from_iterator(
        extrair_texto(),
        vocab_size=50257, # Padrão clássico do GPT-2
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
    )

    # Salva o tokenizador no Drive
    # REMOVE o diretório existente antes de criar novamente para forçar a sincronização
    if os.path.exists(pasta_tok) and os.path.isdir(pasta_tok):
        print(f"Removendo diretório existente do tokenizador: {pasta_tok}")
        shutil.rmtree(pasta_tok)

    os.makedirs(pasta_tok, exist_ok=True) # Cria o diretório novamente
    tokenizer_raw.save_model(pasta_tok)

    # Converte para o formato utilizável pelo Hugging Face Trainer
    tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>", mask_token="<mask>")
    tokenizer.save_pretrained(pasta_tok)
    print(f"Tokenizador salvo com sucesso em: {pasta_tok}")

In [ ]:
import os
import torch
import glob
from datasets import load_dataset, interleave_datasets
from transformers import GPT2LMHeadModel, GPT2TokenizerFast, TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Otimização de memória do PyTorch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
pasta_tok = os.path.join(PASTA_PROJETO, "tokenizador")
pasta_pretreino = os.path.join(PASTA_PROJETO, "checkpoints_pretreino") # Onde busca o pré-treino
pasta_saida_ft = os.path.join(PASTA_PROJETO, "checkpoints_finetuning")  # Nova pasta para o FT

# 1. Carregar Tokenizador e verificar GPU
tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, local_files_only=True)

if torch.cuda.is_available():
    nome_gpu = torch.cuda.get_device_name(0)
    device = torch.device("cuda")
else:
    nome_gpu = "CPU"
    device = torch.device("cpu")
print(f"Executando em: {nome_gpu}")

# 2. Carregar o Modelo Pré-Treinado (Herda o último checkpoint do pré-treino ou o modelo final)
lista_checkpoints = glob.glob(os.path.join(pasta_pretreino, "checkpoint-*"))
if len(lista_checkpoints) > 0:
    ultimo_checkpoint = max(lista_checkpoints, key=os.path.getctime)
    print(f"Carregando base do pré-treino de: {ultimo_checkpoint}")
    model = GPT2LMHeadModel.from_pretrained(ultimo_checkpoint)
else:
    modelo_final_path = os.path.join(PASTA_PROJETO, "modelo_335M_final")
    if os.path.exists(modelo_final_path):
        print(f"Carregando modelo final consolidado de: {modelo_final_path}")
        model = GPT2LMHeadModel.from_pretrained(modelo_final_path)
    else:
        raise FileNotFoundError("Nenhum modelo pré-treinado foi encontrado para iniciar o Fine-Tuning.")

model.to(device)
model.gradient_checkpointing_enable()

# 3. Carregar Datasets de Conversação (Streaming)
ds_en = load_dataset("tatsu-lab/alpaca", split="train", streaming=True)                  # Inglês original
ds_pt = load_dataset("maritaca-ai/mira", split="train", streaming=True)                  # Português
ds_es = load_dataset("bertin-project/alpaca-spanish", split="train", streaming=True)     # Espanhol

# Intercala os datasets mantendo um equilíbrio estável
dataset_conversacao = interleave_datasets([ds_en, ds_pt, ds_es], probabilities=[0.4, 0.3, 0.3], seed=42)

# Prompt template para alinhar conversas no padrão GPT-2
def mapear_prompt_conversacao(exemplo):
    instrucao = exemplo.get('instruction', '')
    contexto = exemplo.get('input', '') if exemplo.get('input') else ''
    resposta = exemplo.get('output', '')

    # Monta a entrada juntando instrução e contexto
    entrada_usuario = f"{instrucao}\n{contexto}".strip() if contexto else instrucao

    # Formato clássico de prompt para GPT-2 (Usa quebras de linha e texto direto)
    texto_formatado = f"User: {entrada_usuario}\nAssistant: {resposta}{tokenizer.eos_token}"
    return {"text_formatted": texto_formatado}

def tokenize_function(examples):
    return tokenizer(examples["text_formatted"], truncation=True, max_length=1024)

# Transforma e tokeniza limpando colunas antigas de forma dinâmica
dataset_mapeado = dataset_conversacao.map(mapear_prompt_conversacao)
tokenized_dataset = dataset_mapeado.map(tokenize_function, batched=True, remove_columns=list(dataset_conversacao.features.keys()) + ["text_formatted"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- CONFIGURAÇÃO ESPECÍFICA PARA AJUSTE FINO NA L4 ---
batch_por_dispositivo = 4
acumulacao_passos = 8        # Lote global = 32
usar_bf16 = True if "L4" in nome_gpu or "A100" in nome_gpu else False
usar_fp16 = not usar_bf16

# 4. Parâmetros de Treinamento de Fine-Tuning
training_args = TrainingArguments(
    output_dir=pasta_saida_ft,
    max_steps=20000,               # Ajuste fino precisa de muito menos passos que o pré-treino
    per_device_train_batch_size=batch_por_dispositivo,
    gradient_accumulation_steps=acumulacao_passos,
    save_steps=1000,
    save_total_limit=2,
    logging_steps=50,
    bf16=usar_bf16,
    fp16=usar_fp16,
    learning_rate=5e-5,            # Taxa de aprendizado bem menor (ex: 5e-5) para não destruir o pré-treino
    weight_decay=0.01,
    warmup_steps=1000,             # Warmup proporcional ao tamanho menor do treino
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print(f"Iniciando AJUSTE FINO (Fine-Tuning) na {nome_gpu}...")
trainer.train()

# Salvar o modelo final ajustado para conversação
model.save_pretrained(os.path.join(PASTA_PROJETO, "modelo_335M_chat_final"))
print("FINE-TUNING CONCLUÍDO COM SUCESSO!")